In [1]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

root
 |-- _c0: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- TP2: double (nullable = true)
 |-- TP3: double (nullable = true)
 |-- H1: double (nullable = true)
 |-- DV_pressure: double (nullable = true)
 |-- Reservoirs: double (nullable = true)
 |-- Oil_temperature: double (nullable = true)
 |-- Motor_current: double (nullable = true)
 |-- COMP: integer (nullable = true)
 |-- DV_electric: integer (nullable = true)
 |-- Towers: integer (nullable = true)
 |-- MPG: integer (nullable = true)
 |-- LPS: integer (nullable = true)
 |-- Pressure_switch: integer (nullable = true)
 |-- Oil_level: integer (nullable = true)
 |-- Caudal_impulses: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- dow: integer (nullable = true)

Tổng số bản ghi: 1516948


In [2]:
# TOP 10 NGÀY CÓ NHIỆT ĐỘ DẦU TRUNG BÌNH CAO NHẤT KHI MÁY VẬN HÀNH
q4_high_temp_days = spark.sql('''
    SELECT
        TO_DATE(timestamp) AS date,
        ROUND(AVG(Oil_temperature), 2) AS nhiet_do_dau_tb,
        ROUND(MAX(Oil_temperature), 2) AS nhiet_do_dau_cao_nhat,
        ROUND(AVG(Motor_current), 2) AS dong_dien_tb,
        COUNT(*) AS so_ban_ghi
    FROM sensor
    WHERE COMP = 0
    GROUP BY TO_DATE(timestamp)
    ORDER BY nhiet_do_dau_tb DESC
    LIMIT 10
''')
# Hiển thị Execution Plan
print("EXECUTION PLAN")
q4_high_temp_days.explain(True)
print("TOP 10 NGÀY CÓ NHIỆT ĐỘ DẦU TRUNG BÌNH CAO NHẤT")
df_q4 = q4_high_temp_days.toPandas()
try:
    display(df_q4)
except NameError:
    print(df_q4.to_string(index=False))

EXECUTION PLAN
== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['nhiet_do_dau_tb DESC NULLS LAST], true
      +- 'Aggregate ['TO_DATE('timestamp)], ['TO_DATE('timestamp) AS date#44, 'ROUND('AVG('Oil_temperature), 2) AS nhiet_do_dau_tb#45, 'ROUND('MAX('Oil_temperature), 2) AS nhiet_do_dau_cao_nhat#46, 'ROUND('AVG('Motor_current), 2) AS dong_dien_tb#47, 'COUNT(1) AS so_ban_ghi#48]
         +- 'Filter ('COMP = 0)
            +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
date: date, nhiet_do_dau_tb: double, nhiet_do_dau_cao_nhat: double, dong_dien_tb: double, so_ban_ghi: bigint
GlobalLimit 10
+- LocalLimit 10
   +- Sort [nhiet_do_dau_tb#45 DESC NULLS LAST], true
      +- Aggregate [to_date(timestamp#1, None, Some(Asia/Saigon), true)], [to_date(timestamp#1, None, Some(Asia/Saigon), true) AS date#44, round(avg(Oil_temperature#7), 2) AS nhiet_do_dau_tb#45, round(max(Oil_temperature#7), 2) AS nhiet_do_dau_cao_nhat#46, round(avg(Motor_current#8)

,date,nhiet_do_dau_tb,nhiet_do_dau_cao_nhat,dong_dien_tb,so_ban_ghi
0,2020-07-15,77.41,89.05,5.74,4166
1,2020-03-29,77.24,83.13,5.63,7199
2,2020-06-07,75.46,77.08,5.48,4888
3,2020-06-06,75.40,78.03,5.50,7343
4,2020-06-05,74.78,77.65,5.53,5546
5,2020-04-18,74.33,78.17,5.65,8592
6,2020-03-12,74.31,78.03,5.80,4765
7,2020-05-20,73.79,76.10,5.48,7268
8,2020-05-30,73.28,76.65,5.65,2901
9,2020-04-12,73.17,76.30,5.81,4553
